In [1]:
import numpy as np
import pandas as pd 
import re


In [2]:
intake_data_raw = pd.read_csv('/kaggle/input/datasets/jackdaoud/animal-shelter-analytics/Austin_Animal_Center_Intakes.csv')
outcome_data_raw = pd.read_csv('/kaggle/input/datasets/jackdaoud/animal-shelter-analytics/Austin_Animal_Center_Outcomes.csv')


In [3]:
def header():
    print('*!*' * 32 + '\n\n\n')
    return

def clean_heders(df) -> pd.DataFrame:
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df

def var_types(df, int_cols, cat_cols, float_cols, datetime_cols, bool_cols) -> pd.DataFrame:
    for col in df.columns:
        if col in int_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
        elif col in cat_cols:
            df[col] = df[col].astype('string')
        elif col in float_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0).astype(float)
        elif col in datetime_cols:
            df[col] = pd.to_datetime(df[col], errors='coerce')
        elif col in bool_cols:
            df[col] = df[col].astype(bool)
    return df

def imported_data_clean(df) -> pd.DataFrame: 
    header()
    df = df[~df.duplicated(subset=['animal_id', 'datetime'])].copy() 
    error_count = 0

    
    return df 

def cln_animal_id(df) -> pd.DataFrame:
    different_len_id = df[df['animal_id'].str.len() != 7]['animal_id'].unique()
    if len(different_len_id) > 0:
        print(f"Found {len(different_len_id)} unique 'animal_id' values with unexpected length:")
        print(different_len_id)
    else:
        print("All 'animal_id' values have the expected length of 7 characters.")
    return df

def datetime_y_apt_id(df, df_name) -> pd.DataFrame:
    if df['datetime'].astype(str).str.contains(r'AM|PM', case = False, regex = True).any():
        df['datetime'] = pd.to_datetime(df['datetime'], format = ('%m/%d/%Y %I:%M:%S %p'), errors = 'coerce') 
        #intake_data['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
    else:
        df['datetime'] = pd.to_datetime(df['datetime'], format = '%Y-%m-%d %H:%M:%S', errors = 'coerce')

    df['std_date_time'] = df['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df['apt_id'] = df['animal_id'].astype(str) + '_' + df['std_date_time'].astype(str)

    if df_name == 'intake':
        df = df.rename(columns = {'datetime': 'datetime_in'})
    else:
        df = df.rename(columns = {'datetime': 'datetime_out'})
    
    return df

def cln_intake_type(df) -> pd.DataFrame:
    df['intake_type'] = df['intake_type'].str.replace(r'.*euth.*', 'qol', regex=True)
   
    #print(f'unique values: {df['intake_type'].unique()}')
    #print(f'\n\n\nnull values: {df['intake_type'].isna().sum()}')

    #this information looks clean and that makes sense as this is recorded by the staff at time of intake
    return df

def intake_condition_clean(df) -> pd.DataFrame:
    medical_conditions = ['sick', 'injured', 'medical', 'aged']
    behavior_conditions = ['feral', 'behavior']
    reproductive_conditions = ['pregnant', 'nursing']
    routine_conditions = ['normal']
    other_conditions = ['other']

    df['pregnant_o_nursing'] = np.where(df['intake_condition'].isin(reproductive_conditions), True, False)

    conditions = [
        df['intake_condition'].isin(medical_conditions) | df['intake_condition'].isin(reproductive_conditions),
        df['intake_condition'].isin(behavior_conditions),
        df['intake_condition'].isin(routine_conditions),
        df['intake_condition'].isin(other_conditions)
    ]

    choices = [
        'medical' ,
        'behavior',
        'routine',
        'other'
    ]

    df['intake_reason'] = np.select(conditions, choices, default = 'unknown')
    return df

def cln_perros_y_gatos(df) -> pd.DataFrame:
    df = df.rename(columns = {
        'animal_type' : 'spp'
    })
    if 'spp' in df.columns:
        df['spp'] = (
            df['spp'].astype(str)
                 .str.strip()
                 .str.lower()
                 .replace({'cat': 'fel', 'dog': 'k9'})
        )
    return df

def cln_spp_wildlife(df) -> pd.DataFrame:

    wildlife_mask = df['intake_type'].str.contains('wildlife', case = False, na = False)
    df.loc[wildlife_mask, 'spp'] = 'wildlife'
    return df

def cln_color(df) -> pd.DataFrame:
    """Clean and standardize intake color descriptions."""

    if 'color' not in df.columns:
        return df

        df['color'] = (
                df['color']
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    .replace({'': np.nan, 'nan': np.nan, 'none': np.nan})
        )

    def normalize_color(value):
        if pd.isna(value):
            return np.nan
        text = str(value).lower().strip()
        text = text.replace('_', ' ')
        text = re.sub(r'[,;&]+', '/', text)
        text = re.sub(r'\b(?:and|&|plus)\b', '/', text)
        text = re.sub(r'\s*/\s*', '/', text)
        text = re.sub(r'[^a-z0-9/ ]+', '', text)
        text = re.sub(r'\s+', ' ', text)
        text = text.strip(' /')
        return text if text else pd.NA

    def detect_pattern(text):
        if not isinstance(text, str) or text == '':
            return pd.NA
        patterns = {
            'tabby': 'tabby',
            'tortie': 'tortie',
            'tortoiseshell': 'tortie',
            'merle': 'merle',
            'brindle': 'brindle',
            'calico': 'calico',
            'point': 'point',
            'smoke': 'smoke',
            'seal': 'seal',
        }
        for key, value in patterns.items():
            if key in text:
                return value
        return np.nan

    def color_components(text):
        if not isinstance(text, str):
            return np.nan, np.nan, np.nan
        if text == '':
            return np.nan, np.nan, np.nan
        parts = [part.strip() for part in text.split('/') if part.strip()]
        if not parts:
            return pd.NA, pd.NA, pd.NA

        primary = parts[0]
        secondary = parts[1] if len(parts) > 1 else pd.NA
        pattern = detect_pattern(text)

        if len(parts) == 1 and isinstance(primary, str) and isinstance(pattern, str):
            primary = re.sub(
                r'\s*(?:tabby|tortie|tortoiseshell|merle|brindle|calico|point|smoke|seal)\b',
                '',
                primary,
            ).strip()
        if isinstance(primary, str) and primary == '':
            primary = np.nan
        if isinstance(secondary, str) and secondary == '':
            secondary = np.nan

        # Promote pattern to primary when no explicit primary is present.
        # Use pd.isna to safely detect missing values (avoids ambiguous boolean checks).
        if pd.isna(primary) and isinstance(pattern, str):
            if pattern == 'tortie':
                primary = 'tortoiseshell'
            else:
                primary = pattern

        return primary, secondary, pattern

    df['cln_color'] = df['color'].apply(normalize_color)
    # coerce any pandas NA to numpy NaN on cln_color
    df['cln_color'] = df['cln_color'].astype(object).where(~pd.isna(df['cln_color']), np.nan)

    comps = df['cln_color'].apply(color_components)
    df[['color_primary', 'color_secondary', 'color_pattern']] = pd.DataFrame(
        comps.tolist(), index=df.index
    )
    # Ensure these new columns contain numpy NaN (not pandas <NA>) for SQL compatibility
    for c in ['color_primary', 'color_secondary', 'color_pattern']:
        df[c] = df[c].astype(object).where(~pd.isna(df[c]), np.nan)

    return df

def cln_spp_other(df: pd.DataFrame) -> pd.DataFrame:
    import re

    patterns = [
        (r"hamster|guinea pig|rat|mouse|gerbil|chinchilla|hedgehog|sugar glider", "sm_rodents_pocket_pets", "rodent"),
        (r"lop", "rabbits_lops", "rabbit"),
        (r"rex", "rabbits_rex", "rabbit"),
        (r"angora|lh|lionhead|wooly", "rabbits_angora_wooly_lionhead", "rabbit"),
        (r"dwarf|hotot|polish|britannia", "rabbits_dwarf", "rabbit"),
        (r"giant", "rabbits_giant", "rabbit"),
        (r"bat|ferret|raccoon|opossum|squirrel|cottontail|prairie dog|armadillo|skunk|fox|coyote|ringtail", "wild_exotic_mammals", "wildlife"),
        (r"snake|python|lizard|turtle|tortoise|frog", "reptiles_amphibians", "reptile"),
        (r"cold water|tropical|crab|tarantula", "aquatic_invertebrates", "aquatic"),
        (r"rabbit|dutch|californian|cinnamon|silver|havana|american|new zealand|english spot|beveren|himalayan|harlequin|rhinelander|belgian hare", "rabbits_standard_other_breeds", "rabbit"),
        (r"bird|avian", "other_birds", "bird"),
        (r"livestock|farm|goat|sheep|cow|pig|llama|alpaca|horse|donkey|mule", "other_livestock", "livestock"),
    ]

    def classify_breed(breed, spp):
        spp_text = str(spp).strip().lower()
        if spp_text in ['k9', 'fel']:
            return breed, spp
        text = str(breed).lower()
        for pattern, clean_breed, clean_spp in patterns:
            if re.search(pattern, text):
                return clean_breed, clean_spp
        return breed, spp

    mapped = df.apply(
        lambda row: classify_breed(row["breed"], row.get("spp", np.nan)),
        axis=1,
        result_type="expand",
    )

    mapped.columns = ["cln_breed", "cln_spp"]
    df = df.copy()
    df[["cln_breed", "cln_spp"]] = mapped
    return df

def cln_k9_breed(df) -> pd.DataFrame:
    k9_mask = df['cln_spp'] == 'k9'
    df.loc[k9_mask, 'is_mix'] = df.loc[k9_mask, 'cln_breed'].str.contains('mix|/', case = False, na = False)
    df['is_mix'] = df['is_mix'].astype('boolean')

    
    df.loc[k9_mask, 'cln_breed'] = df.loc[k9_mask, 'cln_breed'].str.split(r'_mix|_/').str[0]
    
    return df

def cln_akc_group(df) -> pd.DataFrame:
    k9_mask = df['cln_spp'] == 'k9'

    conditions = [
        k9_mask & df['cln_breed'].str.contains('/', na = False),  # Mixes/Hybrids first
        k9_mask & df['cln_breed'].str.contains(
            'retriever|setter|pointer|spaniel|vizsla|weimaraner|brittany|retr|griffon|italiano|span|port_water_dog',
            case = False,
            na = False,
        ),
        k9_mask & df['cln_breed'].str.contains(
            'hound|beagle|basenji|dachshund|whippet|greyhound|rhod|harrier|saluki|pbgv|vendeen|treeing_cur|podengo_pequeno|carolina_dog|jindo|treeing_tennesse_brindle',
            case = False,
            na = False,
        ),
        k9_mask & df['cln_breed'].str.contains(
            'shepherd|collie|corgi|kelpie|cattle_dog|malinois|sheepdog|border|heeler|beauceron|vallhund|briard|tervuren|catahoula|canaan_dog',
            case = False,
            na = False,
        ),
        k9_mask & df['cln_breed'].str.contains(
            r'mastiff|husky|malamute|akita|dane|pyrenees|boxer|rottweiler|pinsch|dogo|mountain_dog|canario|bordeaux|corso|boerboel|st\._bernard|landseer|samoyed|leonberger|newfoundland|entlebucher|akbash|kuvasz|hovawart|kangal|black_mouth_cur|blue_lacy|wolf_hybrid',
            case = False,
            na = False,
        ),
        k9_mask & df['cln_breed'].str.contains(
            'terrier|terr|schnauzer|pinscher|cairn|staffordshire|imaal|dandie|bedlington|sealyham|west_highland|pit_bull|feist', 
            case = False, 
            na = False
        ),
        k9_mask & df['cln_breed'].str.contains(
            'chihuahua|pomeranian|pug|maltese|pekingese|papillon|toy|havanese|bichon|shih_tzu|bruss|crested|chin',
            case = False,
            na = False,
        ),
        k9_mask & df['cln_breed'].str.contains(
            'poodle|bulldog|sharpei|chow|dalmatian|boston|keeshond|lhasa|eskimo|shiba|tulear|schipperke|mexican_hairless|klee_kai|lowchen|finnish_spitz',
            case = False,
            na = False,
        ),
        ~k9_mask  # Non-canine records explicitly caught
    ]

    choices = [
        'mixed_breed',
        'sporting',
        'hound',
        'herding',
        'working',
        'terrier',
        'toy',
        'non_sporting',
        'n/a'
    ]

    df['akc_group'] = np.select(conditions, choices, default = 'unknown_other')

    return df








def cln_fel_breed(df) -> pd.DataFrame:
    """Classify cat breeds into DSH, DMH, DLH, or specific breed if known."""
    
    # Known specific cat breeds
    specific_breeds = {
        'siamese': 'Siamese', 'persian': 'Persian', 'maine coon': 'Maine Coon',
        'bengal': 'Bengal', 'abyssinian': 'Abyssinian', 'ragdoll': 'Ragdoll',
        'sphynx': 'Sphynx', 'devon rex': 'Devon Rex', 'cornish rex': 'Cornish Rex',
        'scottish fold': 'Scottish Fold', 'manx': 'Manx', 'birman': 'Birman',
        'burmese': 'Burmese', 'tonkinese': 'Tonkinese', 'oriental': 'Oriental',
        'british': 'British', 'russian': 'Russian', 'korat': 'Korat',
        'balinese': 'Balinese', 'bombay': 'Bombay', 'japanese': 'Japanese',
        'ragamuffin': 'Ragamuffin', 'somali': 'Somali', 'turkish': 'Turkish',
    }
    
    # Hair length indicators
    longhair_indicators = ['long', 'lh', 'longhair', 'fluffy', 'maine coon', 'persian', 'ragdoll', 'siberian']
    mediumhair_indicators = ['medium', 'mh', 'mediumhair']
    shorthair_indicators = ['short', 'sh', 'shorthair', 'tabby', 'siamese', 'abyssinian']
    
    def classify_fel_breed(breed):
        if pd.isna(breed):
            return 'Unknown'
        
        text = str(breed).lower()
        
        # Check for specific breed first
        for keyword, breed_name in specific_breeds.items():
            if keyword in text:
                return breed_name
        
        # Check hair length if no specific breed found
        if any(indicator in text for indicator in longhair_indicators):
            return 'dlh'
        elif any(indicator in text for indicator in mediumhair_indicators):
            return 'dmh'
        elif any(indicator in text for indicator in shorthair_indicators):
            return 'dsh'
        elif 'mix' in text or 'other' in text:
            # Default to DSH for mixed/other if no hair length specified
            return 'dsh'
        else:
            return 'Unknown'
    
    fel_mask = df['cln_spp'] == 'fel'
    df['cln_breed'] = df['cln_breed'].fillna('')
    df.loc[fel_mask, 'cln_breed'] = df.loc[fel_mask, 'cln_breed'].apply(classify_fel_breed)
    
    return df
   
def age_clean(df, df_name) -> pd.DataFrame:
    df.columns = df.columns.str.replace(r'^age.*$', 'cln_age', regex = True)
    df['age_num'] = df['cln_age'].str.split(r'_| ', n = 1, expand = True)[0]
    df['age_unit'] = df['cln_age'].str.split(r'_| ', n = 1, expand = True)[1]
    #df['age_num'] = df['age_num'].replace(['nan', ''], np.nan)
    #df['cln_age'] = df['cln_age'].replace(['nan', ''], np.nan)
    df['age_num'] = pd.to_numeric(df['age_num'])
    df['age_num'] = df['age_num'].where(df['age_num'] >= 0, np.nan)
    
    conditions = [
        (df['age_unit'].str.contains(r'months?', case = False, na = False, regex = True)), 
        (df['age_unit'].str.contains(r'years?', case = False, na = False, regex = True)),
        (df['age_unit'].str.contains(r'weeks?', case = False, na = False, regex = True)),
        (df['age_unit'].str.contains(r'days?', case = False, na = False, regex = True))
    ]

    choices = [
        (df['age_num'] / 12), #converts the age in months to years
        (df['age_num']), #keeps the same age as it is in years
        (df['age_num'] / 52.1786), #converts from weeks to years
        (df['age_num'] / 365.25) #converts frem days to years
    ]

    df['age_years'] = np.select(conditions, choices, default = np.nan)
    #df = df.drop(columns = ['age_num', 'cln_age'], axis = 1)

    
    if df_name == 'intake':
        df = df.rename(columns = {'age_years': 'age_in_years'})
    else:
        df = df.rename(columns = {'age_years': 'age_out_years'})
    return df

def clean_sex(df, df_name) -> pd.DataFrame:
    df.columns = df.columns.str.replace(r'^sex.*$', 'sex', regex = True)

    condition = [
        df['sex'].str.contains(r'intact_male', case = False, na = False),
        df['sex'].str.contains(r'intact_f*', case = False, na = False),
        df['sex'].str.contains(r'neu*', case = False, na = False),
        df['sex'].str.contains(r'spa*', case = False, na = False),
    ]

    choice = [
        'm',
        'f',
        'n',
        's'
    ]

    df['sex'] = np.select(condition, choice, default = 'unknown')

    if df_name == 'intake':
        df = df.rename(columns = {'sex': 'sex_in'})
    else:
        df = df.rename(columns = {'sex': 'sex_out'})
    return df

def snakecase_data(df) -> pd.DataFrame:
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].str.strip().str.lower().str.replace(' ', '_')
    return df

def table_check(df, df_name, unique_id) -> pd.DataFrame:

    
    header()
    print(f'Beginning errorcheck for: {df_name} ***')
    print(f'\n--- Null Value Check ---')
    for column in df.columns: 
        null_count = df[column].isna().sum() 
        print(f'{column} nulls: {null_count}')
    print('\n')
    
    print(f'--- Integrity Check ---')
    if unique_id in df.columns:
        duplicate_count = df[unique_id].duplicated().sum()
        print(f'Total {unique_id} duplicates: {duplicate_count}')
    else:
        print(f"WARNING: {unique_id} not found for duplication check.")
    print('\n')
    print(f'\n--- Data Type Audit (DF.dtypes) ---')
    print(df.dtypes)
    
    print(f'\n--- Numerical Sanity Check (DF.describe) ---')
    
    numeric_cols = df.select_dtypes(include = ['int64', 'float64', 'Int64', 'int32']).columns
    if not numeric_cols.empty:
        print(df[numeric_cols].describe())
    else:
        print('No standard numerical columns found for description.')

    print(f'\n--- Categorical Value Audit (Top 10 Counts) ---')
    object_cols = df.select_dtypes(include=['object', 'string[python]']).columns

    for column in object_cols:
        print(f'\n{column.upper()}:')
        print(df[column].value_counts().nlargest(10))

    

    print('\n\n\n')

    return df

def clean_outcome_type(df) -> pd.DataFrame:
    unknown_outcome = ['nan', 'missing']
    alive_outcome = ['rto-adopt', 'adoption', 'return to owner']
    administrative_outcome = ['transfer', 'relocate']
    deceased_outcome = ['euthanasia', 'died', 'disposal']
    
    conditions = [
        df['outcome_type'].isin(unknown_outcome),
        df['outcome_type'].isin(alive_outcome),
        df['outcome_type'].isin(administrative_outcome),
        df['outcome_type'].isin(deceased_outcome)
    ]

    choices = [
        'unknown',
        'alive',
        'admin',
        'deceased'
    ]

    df['outcome_category'] = np.select(conditions, choices, default = 'unknown')
    df = df.drop(columns = 'outcome_type', axis = 1)
    return df

def clean_outcome_subtype(df) -> pd.DataFrame:
    location_subtype = ['in kennel', 'offsite', 'at vet', 'barn', 'enroute', 'in surgery'] #the animal's specific physical location or temporary status
    behavior_subtype = ['suffering', 'medical', 'aggressive', 'rabies risk', 'behavior'] #indicates the reason for a specific outcome (often euthanasia or specialized treatment)
    program_subtype = ['partner', 'underage', 'foster', 'in foster', 'snr', 'scrp', 'prc'] #subtypes related to specific shelter programs or transfer partners
    admin_subtype = ['field', 'possible theft', 'customer s', 'court/investigation', 'emer'] #Miscellaneous or administrative details
    unknown_subtype = ['nan']

    conditions = [
        df['outcome_subtype'].isin(location_subtype),
        df['outcome_subtype'].isin(behavior_subtype),
        df['outcome_subtype'].isin(program_subtype),
        df['outcome_subtype'].isin(admin_subtype),
        df['outcome_subtype'].isin(unknown_subtype)
    ]

    choices = [
        'location' ,
        'behavior',
        'program',
        'admin',
        'unknown'
    ]

    df['outcome_subcategory'] = np.select(conditions, choices, default = 'unknown')
    df = df.drop(columns = 'outcome_subtype', axis = 1)
    return df

In [4]:
def intake_line(df) -> pd.DataFrame:
    print('\n\n\n')
    header()
    print(f'Beginning to create intake table')
    return (df
            .pipe(clean_heders)
            .pipe(imported_data_clean)
            .pipe(cln_animal_id)
            .pipe(datetime_y_apt_id, df_name = 'intake')
            .pipe(snakecase_data)
            .pipe(cln_color)
            .pipe(cln_intake_type)
            .pipe(intake_condition_clean)
            .pipe(cln_perros_y_gatos)
            .pipe(cln_spp_wildlife)
            .pipe(cln_spp_other)
            .pipe(cln_k9_breed)
            .pipe(cln_akc_group)
            .pipe(cln_fel_breed)
            .pipe(age_clean, df_name = 'intake')
            .pipe(clean_sex, df_name = 'intake')
            .pipe(lambda row: row.drop(columns = ['name', 'monthyear', 'found_location', 'std_date_time', 'breed', 'cln_age', 'color', 'cln_color', 
                                                  #'primary_breed', 
                                                  'age_unit', 'age_num'], axis = 1))
            .pipe(var_types, 
                  int_cols = [], 
                  cat_cols = ['animal_id', 'intake_type', 'intake_condition', 'spp', 'sex_in',
                            'color', 'apt_id', 'cln_color', 'color_primary', 'color_secondary', 'color_pattern',
                            'intake_reason', 'cln_breed', 'cln_spp', 'primary_breed', 'akc_group', 'cln_breed', 'age_unit'], 
                  float_cols = ['cln_age', 'age_years'],
                  datetime_cols = ['datetime_in'],
                  bool_cols = ['pregnant_o_nursing', 'mixed_breed']
                  )
            .pipe(table_check, df_name = 'intake', unique_id = 'animal_id')
        
            
            )
def outcome_line(df) -> pd.DataFrame:
    print('\n\n\n')
    header()
    print(f'Beginning to create outcome table')
    
    return (
        df
        .pipe(clean_heders)
        .pipe(imported_data_clean)
        .pipe(datetime_y_apt_id, df_name = 'outcome')
        .pipe(snakecase_data)
        .pipe(cln_perros_y_gatos)
        .pipe(clean_sex, df_name = 'outcome')
        .pipe(age_clean, df_name = 'outcome')
        .pipe(clean_outcome_type)
        .pipe(clean_outcome_subtype)
        .pipe(lambda row: row.drop(columns = ['name', 'monthyear', 'date_of_birth', 'color', 'breed', 'cln_age', 'color', 'age_unit', 'age_num'], axis = 1))
        .pipe(table_check, df_name = 'outcome', unique_id = 'animal_id')
    )



In [5]:
intake_data = intake_line(intake_data_raw)
print(intake_data.columns)
# print('\n\n\n')
#print(intake_data['cln_spp'].unique())





*!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!*



Beginning to create intake table
*!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!*



All 'animal_id' values have the expected length of 7 characters.
*!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!*



Beginning errorcheck for: intake ***

--- Null Value Check ---
animal_id nulls: 0
datetime_in nulls: 0
intake_type nulls: 0
intake_condition nulls: 0
spp nulls: 0
sex_in nulls: 0
apt_id nulls: 0
color_primary nulls: 0
color_secondary nulls: 57836
color_pattern nulls: 89890
pregnant_o_nursing nulls: 0
intake_reason nulls: 0
cln_breed nulls: 0
cln_spp nulls: 0
is_mix nulls: 53661
akc_group nulls: 0
age_in_years nulls: 7


--- Integrity Check ---
Total animal_id duplicates: 13171



--- Data Type Audit (DF.dtypes) ---
animal_id             string[python]
datetime_in           datetime64[ns]
int

In [6]:
outcome_data = outcome_line(outcome_data_raw)
print(outcome_data.head())





*!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!*



Beginning to create outcome table
*!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!*



*!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!**!*



Beginning errorcheck for: outcome ***

--- Null Value Check ---
animal_id nulls: 0
datetime_out nulls: 0
spp nulls: 0
sex_out nulls: 0
std_date_time nulls: 0
apt_id nulls: 0
age_out_years nulls: 12
outcome_category nulls: 0
outcome_subcategory nulls: 0


--- Integrity Check ---
Total animal_id duplicates: 13165



--- Data Type Audit (DF.dtypes) ---
animal_id                      object
datetime_out           datetime64[ns]
spp                            object
sex_out                        object
std_date_time                  object
apt_id                         object
age_out_years                 float64
outcome_category               object
outcom

In [7]:
intake_data.to_csv('intake_data.csv', index = False)
outcome_data.to_csv('outcome_data.csv', index = False)

In [8]:
!pip freeze > requirements.txt